# Syllable correlates of LD2–LD7 (supplementary)

fig_ld1 panel B for the next discriminants: per (trial bin, syllable), the Pearson correlation across sessions between the session's occupancy of that syllable in that bin (the LDA feature) and the session's score on LDk. Same data, same arithmetic and the same reading as fig_ld1 B: paw states, whisking and licking in one panel (whisking and licking are the dashed grey lines); grey band = the permutation null (scores shuffled across sessions, 2.5–97.5 %, widest over the features drawn), a line is bold where it leaves the null and faint inside it; descriptive, no correction across cells. Panel titles give each LD's share of the between-mouse variance.

**Read with:** an LD's sign is arbitrary (flip it and every line flips), and beyond LD1–LD2 the individual axes are not stable under resampling sessions — they rotate among themselves inside a stable subspace (`4_mice/lda_shrinkage_figures`, axis-stability supplement). So these panels describe what each axis of *this* fit loads on, not fixed behavioural dimensions.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
for p in [str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
try:
    import numpy._core.numeric  # noqa: F401
except ModuleNotFoundError:     # the LDA file was pickled under numpy 2
    import numpy.core.numeric, numpy.core.multiarray
    sys.modules['numpy._core.numeric'], sys.modules['numpy._core.multiarray'] = np.core.numeric, np.core.multiarray
import paper_style as ps
import fig_layout as fl
ps.use('paper')
ps.use_paw_states('uniform')
FIG = 'fig_ld_syllables'

LDA_FILE = PI / 'clustering' / 'data_files' / 'mouse_LDA_5_bins_raw_shrink0.5_wmouse_360_02-10-2026'   # as fig_ld1
SYLLABLE_FILE = next(f for f in [PI / 'data' / '8_k_10_bin_syllables_02-10-2026',                  # the file the LDA was fit on
                                 PI / 'clustering' / 'data_files' / '8_k_10_bin_syllables_02-10-2026'] if f.exists())
LDS = [2, 3, 4, 5, 6, 7]
N_PERMS = 1000          # as fig_ld1
SEED = 0

## Session occupancy (fig_ld1 cell 9) and each LD's share of the between-mouse variance

In [ ]:
lda = pd.read_pickle(LDA_FILE)
n_axes = sum(isinstance(c, (int, np.integer)) for c in lda.columns)
between = lda.groupby('mouse_name')[list(range(n_axes))].mean().var(axis=0, ddof=0).to_numpy()
AXIS_SHARE = between / between.sum()

syl = pd.read_parquet(SYLLABLE_FILE)
syl['session'] = syl['sample'].str[:36]
syl = syl[syl['session'].isin(lda['session'])]
EPOCHS = ps.EPOCH_NAMES
trials = (syl.pivot(index=['mouse_name', 'session', 'sample'], columns='broader_label', values='binned_sequence')
          .reset_index().dropna(subset=EPOCHS))
codes = ps.paw_codes(np.vstack([np.hstack(r) for r in trials[EPOCHS].to_numpy()]).astype(float))
paw, whisk, lick = ps.decode_syllables(codes)
OCC = np.stack([np.where(np.isnan(paw), np.nan, paw == k) for k in range(ps.N_PAW_STATES)] + [whisk, lick], axis=-1)
sess_idx = trials['session'].to_numpy()
order = lda['session'].to_numpy()
with np.errstate(invalid='ignore'):
    SESS_OCC = np.stack([np.nanmean(OCC[sess_idx == s], axis=0) for s in order])   # (sessions, 40, 10)
N_BINS = SESS_OCC.shape[1]
FEAT_LABEL = [ps.paw_label(k) for k in range(ps.N_PAW_STATES)] + ['Whisk', 'Lick']
FEAT_COLORS = ps.SYLLABLE_COLORS
print(f'{SYLLABLE_FILE.name}: {len(order)} sessions x {N_BINS} bins x {SESS_OCC.shape[2]} syllables; '
      f'between-mouse share LD2-7: {np.round(AXIS_SHARE[1:7], 3)}')

## Structure coefficients and their nulls

In [ ]:
def colwise_r(Mx, v):
    Mc = Mx - np.nanmean(Mx, axis=0)
    Mc = np.nan_to_num(Mc)
    vc = v - v.mean()
    return (Mc.T @ vc) / (np.sqrt((Mc ** 2).sum(axis=0)) * np.sqrt((vc ** 2).sum()))

flat = SESS_OCC.reshape(len(order), -1)
STRUCT, NULL_LO, NULL_HI = {}, {}, {}
for k in LDS:
    score = lda[k - 1].to_numpy(float)
    STRUCT[k] = colwise_r(flat, score).reshape(N_BINS, -1)
    rng = np.random.default_rng(SEED + k)
    null = np.stack([colwise_r(flat, rng.permutation(score)).reshape(N_BINS, -1) for _ in range(N_PERMS)])
    NULL_LO[k], NULL_HI[k] = np.nanpercentile(null, [2.5, 97.5], axis=0)
    out = (STRUCT[k] < NULL_LO[k]) | (STRUCT[k] > NULL_HI[k])
    print(f'LD{k}: {out.sum()} of {out.size} cells outside the null; strongest |r| = {np.nanmax(np.abs(STRUCT[k])):.2f}')

## The figure

In [ ]:
OUTLINE = [pe.Stroke(linewidth=plt.rcParams['lines.linewidth'] + 0.9, foreground='0.35'), pe.Normal()]
YLIM = (-0.75, 0.75)
LS = {f: ((0, (3, 1.5)) if f >= 8 else '-') for f in range(10)}   # whisk / lick dashed: whisk's grey is close to paw 1's

def draw_ld(fig, spec, k, show_epochs=True, ylabel=True):
    # paw states, whisking and licking in one panel (whisk / lick are the grey lines)
    ax = fig.add_subplot(spec)
    x = np.arange(N_BINS)
    feats = list(range(ps.N_PAW_STATES)) + [8, 9]
    ax.fill_between(x, NULL_LO[k][:, feats].min(1), NULL_HI[k][:, feats].max(1), color=ps.NULL_BAND, alpha=0.3, lw=0)
    for f in feats:
        y = STRUCT[k][:, f]
        ls = LS[f]
        ax.plot(x, y, color=FEAT_COLORS[f], alpha=0.3, ls=ls)
        ax.plot(x, np.where((y >= NULL_LO[k][:, f]) & (y <= NULL_HI[k][:, f]), np.nan, y), color=FEAT_COLORS[f],
                label=FEAT_LABEL[f], path_effects=OUTLINE, zorder=3 if f >= 8 else 2, ls=ls)
    ax.axhline(0, color=ps.NEUTRAL, lw=0.6)
    ps.epoch_lines(ax, N_BINS, label=show_epochs, names=ps.EPOCH_SHORT)
    ax.set_xlim(0, N_BINS - 1); ax.set_xticks([]); ax.set_ylim(*YLIM)
    if ylabel:
        ax.set_ylabel('r with LD (sessions)')
    ax.set_title(f'LD{k}  ({100 * AXIS_SHARE[k - 1]:.0f}% of between-mouse var.)', pad=14 if show_epochs else 4,
                 fontweight='bold', fontsize=plt.rcParams['font.size'])
    return [ax]

fig = plt.figure(figsize=(fl.FIG_WIDTH, 5.6))
G = fig.add_gridspec(3, 2, wspace=0.14, hspace=0.6, left=0.08, right=0.99, top=0.93, bottom=0.12)
panels = {}
for i, k in enumerate(LDS):
    r_, c_ = divmod(i, 2)
    panels[k] = draw_ld(fig, G[r_, c_], k, ylabel=(c_ == 0))
    if c_:
        panels[k][0].tick_params(labelleft=False)
kw = dict(frameon=False, loc='upper center', handlelength=1.2, columnspacing=0.8,
          fontsize=plt.rcParams['legend.fontsize'] * 0.9)
from matplotlib.lines import Line2D
h = [Line2D([], [], color=FEAT_COLORS[f], label=FEAT_LABEL[f], path_effects=OUTLINE, ls=LS[f]) for f in range(10)]
fig.legend(handles=h, ncol=10, bbox_to_anchor=(0.53, 0.06), **kw)
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))